<a href="https://colab.research.google.com/github/Mahendra2409/PyBlender/blob/main/Colab_Script/drive_to_gcs_sync.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
<a href="https://drive.google.com/drive/folders/1sj-RqD5HRypGx-ZLqXpvyzY1CN84qJu-?usp=drive_link" target="_parent"><img src="https://img.shields.io/badge/PyBlender_Render_Farm-blue?logo=googledrive&logoColor=white" alt="PyBlender_Render_Farm"/></a>

# ☁️ Drive → GCS Sync

Recursively syncs the entire `PointCloud/` folder from Google Drive to GCS.

**Only uploads new files** — safe to re-run anytime to keep GCS up-to-date.

> Uses a **service account key** stored in **Colab Secrets** for GCS auth (bucket is on a different Google account).
> 
> To add the secret: click 🔑 icon in left sidebar → **+ Add new secret** → name it `GCS_SERVICE_ACCOUNT_KEY` → paste the full JSON content of your service account key.

In [ ]:
#@title 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title 2. Install GCS SDK
!pip install google-cloud-storage -q

In [ ]:
#@title 3. Authenticate GCS (Service Account via Colab Secrets)
import os
from google.colab import userdata

# Load service account key from Colab Secrets
gcs_key_json = userdata.get('GCS_SERVICE_ACCOUNT_KEY')

GCS_KEY_PATH = "/tmp/gcs_service_account.json"
with open(GCS_KEY_PATH, "w") as f:
    f.write(gcs_key_json)

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = GCS_KEY_PATH
print(f"✅ Service account key loaded from Colab Secrets")
print(f"   Saved to {GCS_KEY_PATH}")

# Verify connection
from google.cloud import storage
client = storage.Client()
bucket = client.bucket("pyblender-render-farm")
try:
    next(bucket.list_blobs(max_results=1), None)
    print(f"✅ Connected to gs://pyblender-render-farm")
except Exception as e:
    print(f"❌ Connection failed: {e}")

In [ ]:
#@title 4. Sync Config

SYNC_CONFIG = {
    # --- Source: folder on Google Drive to sync ---
    "DRIVE_SYNC_DIR": "/content/drive/MyDrive/PyBlender_Render_Farm/PointCloud",

    # --- Destination: GCS prefix (mirrors the Drive folder structure) ---
    "GCS_PREFIX": "PointCloud",

    # --- GCS Bucket ---
    "GCS_BUCKET": "pyblender-render-farm",
}

print(f"Drive:  {SYNC_CONFIG['DRIVE_SYNC_DIR']}")
print(f"GCS:    gs://{SYNC_CONFIG['GCS_BUCKET']}/{SYNC_CONFIG['GCS_PREFIX']}/")

# Preview what will be synced
import os
src = SYNC_CONFIG["DRIVE_SYNC_DIR"]
if os.path.exists(src):
    total = sum(len(files) for _, _, files in os.walk(src))
    print(f"\nFound {total} files across:")
    for d in sorted(next(os.walk(src))[1]):
        sub = os.path.join(src, d)
        for sd in sorted(next(os.walk(sub))[1]):
            count = len(os.listdir(os.path.join(sub, sd)))
            print(f"  {d}/{sd}/ ({count} files)")
else:
    print(f"\n❌ Directory not found: {src}")

In [ ]:
#@title 5. Run Sync (Drive → GCS)
import os
from google.cloud import storage

client = storage.Client()
bucket = client.bucket(SYNC_CONFIG["GCS_BUCKET"])

drive_root = SYNC_CONFIG["DRIVE_SYNC_DIR"]
gcs_prefix = SYNC_CONFIG["GCS_PREFIX"]

# Build set of existing GCS blobs for fast skip
print(f"Scanning existing GCS files under {gcs_prefix}/...")
existing_blobs = set()
for blob in bucket.list_blobs(prefix=gcs_prefix + "/"):
    existing_blobs.add(blob.name)
print(f"  {len(existing_blobs)} files already on GCS\n")

total_uploaded = 0
total_skipped = 0

for root, dirs, files in os.walk(drive_root):
    for fname in sorted(files):
        local_path = os.path.join(root, fname)
        relative = os.path.relpath(local_path, drive_root)
        gcs_path = f"{gcs_prefix}/{relative}"

        if gcs_path in existing_blobs:
            total_skipped += 1
            continue

        blob = bucket.blob(gcs_path)
        blob.upload_from_filename(local_path)
        size_mb = os.path.getsize(local_path) / (1024 * 1024)
        print(f"  ✅ {gcs_path} ({size_mb:.1f} MB)")
        total_uploaded += 1

print(f"\n{'='*50}")
print(f"  SYNC COMPLETE")
print(f"  Uploaded: {total_uploaded}")
print(f"  Skipped:  {total_skipped}")
print(f"{'='*50}")

In [ ]:
#@title 6. Verify: List all files on GCS
gcs_prefix = SYNC_CONFIG["GCS_PREFIX"]
blobs = list(bucket.list_blobs(prefix=gcs_prefix + "/"))

from collections import defaultdict
folders = defaultdict(list)
for b in blobs:
    parts = b.name.split("/")
    if len(parts) >= 3:
        folder_key = "/".join(parts[:3])
    else:
        folder_key = "/".join(parts[:-1])
    folders[folder_key].append(b)

print(f"gs://{SYNC_CONFIG['GCS_BUCKET']}/{gcs_prefix}/\n")
for folder in sorted(folders.keys()):
    files = folders[folder]
    total_mb = sum((b.size or 0) for b in files) / (1024 * 1024)
    print(f"  {folder}/ — {len(files)} files ({total_mb:.1f} MB)")

print(f"\nTotal: {len(blobs)} files")

In [ ]:
#@title 7. Sync GCS RenderImages → Drive
import os
from google.cloud import storage

# ==========================================
# CONFIG: GCS → Drive
# ==========================================
GCS_RENDER_PREFIX = "RenderImages"  # GCS folder to download from
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/PyBlender_Render_Farm/Kaggle_Output"  # Drive destination

client = storage.Client()
bucket = client.bucket("pyblender-render-farm")

blobs = list(bucket.list_blobs(prefix=GCS_RENDER_PREFIX + "/"))
print(f"Found {len(blobs)} files on GCS under {GCS_RENDER_PREFIX}/\n")

downloaded = 0
skipped = 0

for blob in blobs:
    # Skip "directory" blobs
    if blob.name.endswith("/"):
        continue

    # Mirror GCS structure: RenderImages/x/y.png -> Kaggle_Output/x/y.png
    relative = os.path.relpath(blob.name, GCS_RENDER_PREFIX)
    dest_path = os.path.join(DRIVE_OUTPUT_DIR, relative)

    if os.path.exists(dest_path):
        skipped += 1
        continue

    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    blob.download_to_filename(dest_path)
    downloaded += 1
    print(f"  ✅ {relative}")

print(f"\n{'='*50}")
print(f"  GCS → DRIVE SYNC COMPLETE")
print(f"  Downloaded: {downloaded}")
print(f"  Skipped:    {skipped}")
print(f"  Dest:       {DRIVE_OUTPUT_DIR}")
print(f"{'='*50}")
